## LG Aimers 9기 — 피처 묶음별 forward CV (Colab)

첫 제출의 **LightGBM 65% + CatBoost 35% logit 앙상블**을 고정한 채, 새 피처만 묶음별로 바꿔 효과를 분리해서 확인한다. 검증은 2022·2023·2024 forward CV이며, 최종 판단은 2023·2024 pooled Brier/BSS와 두 시즌의 방향 일치 여부를 함께 본다. 이 노트북은 제출 ZIP을 만들지 않는다.

### 0. Drive 마운트와 프로젝트 확인

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Colab이 아니므로 Drive 마운트를 건너뜁니다.")

candidates = [
    Path("/content/drive/MyDrive/LG_aimers_9th"),
    Path("/content/drive/MyDrive/LG Aimers 9th"),
    Path.cwd(),
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "experiments/05_feature_ablation_cv.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "LG_aimers_9th 폴더 또는 experiments/05_feature_ablation_cv.py를 찾지 못했습니다."
    )

required = [
    PROJECT_ROOT / "data/train.csv",
    PROJECT_ROOT / "src/features.py",
    PROJECT_ROOT / "src/evaluation.py",
    PROJECT_ROOT / "src/metrics.py",
    PROJECT_ROOT / "experiments/05_feature_ablation_cv.py",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"필수 파일이 없습니다: {missing}")

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd

print("PROJECT_ROOT:", PROJECT_ROOT)
print("train.csv size (GB):", round((PROJECT_ROOT / "data/train.csv").stat().st_size / 1e9, 3))

### 1. 모델 패키지와 GPU 확인

Colab 기본 NumPy·pandas를 다시 설치하지 않고 LightGBM과 CatBoost만 설치한다. A100은 CatBoost에 사용하고 LightGBM은 CPU로 학습한다. 설치 직후 import 오류가 생기면 `런타임 다시 시작` 후 이 셀부터 다시 실행한다.

In [ ]:
install = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "-q",
        "lightgbm==4.6.0", "catboost==1.2.8",
    ],
    capture_output=True, text=True,
)
if install.returncode != 0:
    print(install.stdout)
    print(install.stderr)
    raise RuntimeError("모델 패키지 설치에 실패했습니다. 위 상세 로그를 확인하세요.")

import lightgbm
import catboost

gpu_binary = shutil.which("nvidia-smi")
gpu_check = (
    subprocess.run([gpu_binary, "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
    if gpu_binary else None
)
gpu_available = gpu_check is not None and gpu_check.returncode == 0
print("LightGBM:", lightgbm.__version__)
print("CatBoost:", catboost.__version__)
print("GPU:", gpu_check.stdout.strip() if gpu_available else "not detected — CPU fallback")

### 2. 실험 설정

`base`는 첫 제출과 같은 피처다. `base_no_game_type`은 R-only LightGBM에서 상수인 `game_type`만 제거한다. 나머지는 rate 차이, 최근 흐름, pitch-mix 형태, 상황 상호작용, 팀 관점 문맥을 각각 단독으로 추가하며 `all_new`에서 모두 결합한다. 모델 파라미터와 65:35 혼합비는 고정한다.

In [ ]:
VALID_SEASONS = [2022, 2023, 2024]
CONFIGS = [
    "base",
    "base_no_game_type",
    "rate_gaps",
    "recent_dynamics",
    "pitchmix_shape",
    "context_interactions",
    "team_context",
    "all_new",
]
THREADS = max(1, min(8, os.cpu_count() or 2))
LGBM_ESTIMATORS = 500
CATBOOST_ITERATIONS = 85
CATBOOST_TASK_TYPE = "GPU" if gpu_available else "CPU"
FORCE_RERUN = False

print("configs:", CONFIGS)
print("folds:", VALID_SEASONS)
print("CatBoost task:", CATBOOST_TASK_TYPE, "| threads:", THREADS)
print("unique model fits:", len(CONFIGS) * len(VALID_SEASONS) * 2 - len(VALID_SEASONS))
print("FORCE_RERUN:", FORCE_RERUN)

### 3. 피처 ablation forward CV 실행

각 fold는 검증 시즌보다 이전 시즌만 학습한다. LightGBM은 기존과 같이 R 경기만, CatBoost는 R·F 모두 사용한다. 새 피처는 모두 현재 한 행의 공식 as-of 값만 이용하며, 실행 초기에 행 순서 변경·분할·단일 행 변환이 동일한지 자동 검사한다. 완료된 fold 예측·중요도는 Drive에 캐시된다.

In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "outputs/feature_ablation_v1"
cmd = [
    sys.executable, str(PROJECT_ROOT / "experiments/05_feature_ablation_cv.py"),
    "--train-path", str(PROJECT_ROOT / "data/train.csv"),
    "--output-dir", str(OUTPUT_ROOT),
    "--valid-seasons", *map(str, VALID_SEASONS),
    "--configs", *CONFIGS,
    "--threads", str(THREADS),
    "--lgbm-estimators", str(LGBM_ESTIMATORS),
    "--catboost-iterations", str(CATBOOST_ITERATIONS),
    "--catboost-task-type", CATBOOST_TASK_TYPE,
]
if FORCE_RERUN:
    cmd.append("--force-rerun")

print("실험을 시작합니다. 런타임이 끊기면 FORCE_RERUN=False로 같은 셀을 다시 실행하세요.")
subprocess.run(cmd, check=True)

### 4. 실행 결과와 안전성 검사 확인

In [ ]:
latest_path = OUTPUT_ROOT / "latest_run.json"
if not latest_path.exists():
    raise FileNotFoundError("latest_run.json이 없습니다. 3번 실행 셀을 먼저 완료하세요.")
with latest_path.open(encoding="utf-8") as handle:
    latest = json.load(handle)
RUN_DIR = Path(latest["run_dir"])
with (RUN_DIR / "summary.json").open(encoding="utf-8") as handle:
    summary = json.load(handle)
with (RUN_DIR / "row_independence_audit.json").open(encoding="utf-8") as handle:
    audit = json.load(handle)

print("RUN_DIR:", RUN_DIR)
print("row-independence audit:", audit)
if audit["status"] != "passed":
    raise RuntimeError("행 독립성 검사를 통과하지 못했습니다.")
print("selected_config:", summary["selected_config"])
print("safe_improving_configs:", summary["safe_improving_configs"])

### 5. 65:35 고정 앙상블의 피처별 핵심 비교

Brier와 `delta_*_brier`는 낮을수록 좋고, BSS는 높을수록 좋다. 자동 채택 후보는 2023·2024 pooled가 base보다 좋으면서 두 시즌 어느 쪽도 악화시키지 않은 설정뿐이다. 작은 차이는 바로 제출하지 않고 이후 반복·튜닝 실험에서 재확인한다.

In [ ]:
config_summary = pd.read_csv(RUN_DIR / "config_summary.csv")
blend_summary = config_summary[
    config_summary["model"].eq("fixed_logit_65_35")
].sort_values(["recent_brier", "worst_recent_fold_brier"])
display(blend_summary[[
    "config", "recent_brier", "recent_bss_raw", "recent_mean_gap",
    "delta_recent_brier_vs_base", "delta_recent_bss_raw_vs_base",
    "delta_brier_2023_vs_base", "delta_brier_2024_vs_base",
    "improves_recent", "safe_2023_2024",
]].reset_index(drop=True))

### 6. 모델별 효과와 `game_type` 제거 대조

같은 피처가 LightGBM·CatBoost 중 어디에 실제로 기여했는지 확인한다. `base_no_game_type`에서 CatBoost는 원래부터 `game_type`을 쓰지 않으므로 base 예측을 그대로 재사용했고, 차이는 LightGBM 입력에서만 발생한다.

In [ ]:
display(config_summary[[
    "config", "model", "recent_brier", "recent_bss_raw",
    "delta_recent_brier_vs_base", "delta_brier_2023_vs_base",
    "delta_brier_2024_vs_base",
]].sort_values(["model", "recent_brier"]).reset_index(drop=True))

game_type_control = config_summary[
    config_summary["config"].isin(["base", "base_no_game_type"])
][
    ["config", "model", "recent_brier", "delta_recent_brier_vs_base"]
]
display(game_type_control.reset_index(drop=True))

### 7. 폴드별 안정성과 R·F 성능

최근 pooled만 좋아도 한 시즌에서 크게 악화되면 보류한다. F 경기는 분포가 다르고 LightGBM 학습에서 제외되므로, 전체 점수와 함께 R·F 방향을 별도로 본다.

In [ ]:
fold_metrics = pd.read_csv(RUN_DIR / "fold_metrics.csv")
blend_folds = fold_metrics[
    fold_metrics["model"].eq("fixed_logit_65_35")
    & fold_metrics["partition"].eq("fold")
    & fold_metrics["scope"].eq("all")
]
fold_table = blend_folds.pivot(index="config", columns="valid_season", values="brier")
display(fold_table.sort_values(2024))

type_metrics = fold_metrics[
    fold_metrics["model"].eq("fixed_logit_65_35")
    & fold_metrics["partition"].eq("recent_pooled")
    & fold_metrics["scope"].isin(["R", "F"])
]
display(type_metrics[[
    "config", "scope", "n", "brier", "bss_raw",
    "target_rate", "prediction_mean", "mean_gap",
]].sort_values(["scope", "brier"]).reset_index(drop=True))

### 8. 선택 설정의 취약 구간

표본 1,000행 이상인 구간만 표시한다. 하위 집단 BSS는 공식 점수처럼 합산하는 값이 아니라, cold start·선수 이력·월별 구간 중 어디에서 실패하는지 찾는 진단값이다.

In [ ]:
SELECTED_CONFIG = summary["selected_config"]
groups = pd.read_csv(RUN_DIR / "group_metrics.csv")
weak = groups[
    groups["config"].eq(SELECTED_CONFIG)
    & groups["model"].eq("fixed_logit_65_35")
    & groups["valid_season"].isin([2023, 2024])
    & ~groups["group"].eq("overall")
    & groups["n"].ge(1000)
].sort_values(["valid_season", "bss_raw"])
display(weak[[
    "valid_season", "group", "value", "n", "brier",
    "bss_raw", "roc_auc", "target_rate", "prediction_mean", "mean_gap",
]].groupby("valid_season", group_keys=False).head(15).reset_index(drop=True))

### 9. 보정 상태와 모델 다양성

보정 표는 확률 구간별 예측 평균과 실제 성공률 차이를 보여준다. 여기서는 calibrator를 학습하지 않고 진단만 한다. LightGBM·CatBoost 상관도가 너무 높으면 피처가 좋아져도 앙상블 추가 이득은 제한될 수 있다.

In [ ]:
calibration = pd.read_csv(RUN_DIR / "calibration_tables.csv")
selected_calibration = calibration[
    calibration["config"].eq(SELECTED_CONFIG)
    & calibration["model"].eq("fixed_logit_65_35")
    & calibration["partition"].eq("recent_pooled")
    & calibration["scope"].eq("all")
].copy()
selected_calibration["calibration_gap"] = (
    selected_calibration["prediction_mean"] - selected_calibration["target_rate"]
)
ece = np.average(
    selected_calibration["calibration_gap"].abs(),
    weights=selected_calibration["n"],
)
print("selected recent diagnostic ECE:", round(float(ece), 8))
display(selected_calibration[[
    "bin_index", "n", "prediction_mean", "target_rate", "calibration_gap"
]])

correlations = pd.read_csv(RUN_DIR / "prediction_correlations.csv")
display(correlations[
    correlations["config"].eq(SELECTED_CONFIG)
].reset_index(drop=True))

### 10. 새 피처의 사용 여부와 중요도

피처 수만 늘어난 것이 아니라 실제 split/gain에 사용됐는지 모델별로 확인한다. LightGBM gain과 CatBoost importance의 절대 크기는 서로 비교하지 않고, 같은 모델 안에서 순위만 본다.

In [ ]:
manifest = pd.read_csv(RUN_DIR / "feature_manifest.csv")
present_manifest = manifest[manifest["position"].ge(0)]
feature_counts = present_manifest.groupby(["config", "model"]).agg(
    feature_count=("feature", "size"),
    added_vs_base=("is_added_vs_base", "sum"),
).reset_index()
display(feature_counts)

importance = pd.read_csv(RUN_DIR / "feature_importance.csv")
selected_importance = (
    importance[importance["config"].eq(SELECTED_CONFIG)]
    .groupby(["model", "feature"], as_index=False)["importance"].mean()
)
selected_added = present_manifest[
    present_manifest["config"].eq(SELECTED_CONFIG)
    & present_manifest["is_added_vs_base"]
][["model", "feature"]].drop_duplicates()
selected_added_importance = selected_importance.merge(
    selected_added, on=["model", "feature"], how="inner"
)
for model_name in ["lightgbm", "catboost"]:
    print(f"\n{model_name} — 전체 중요도 상위 20")
    display(
        selected_importance[selected_importance["model"].eq(model_name)]
        .nlargest(20, "importance").reset_index(drop=True)
    )
    print(f"{model_name} — 선택 설정에서 새로 추가된 피처 중요도")
    display(
        selected_added_importance[selected_added_importance["model"].eq(model_name)]
        .sort_values("importance", ascending=False).reset_index(drop=True)
    )

### 11. 자동 판정 해석

자동 선택은 실험 결과를 누락 없이 정리하기 위한 안전장치다. `safe_improving_configs`가 비어 있으면 새 피처가 안정적으로 개선되지 않았다는 뜻이므로 base를 유지한다. 후보가 있어도 다음 단계에서 반복 seed 또는 파라미터 실험으로 재현성을 확인한 뒤 04 최종 학습 코드에 반영한다.

In [ ]:
base_row = blend_summary[blend_summary["config"].eq("base")].iloc[0]
selected_row = blend_summary[blend_summary["config"].eq(SELECTED_CONFIG)].iloc[0]
decision = {
    "selected_config": SELECTED_CONFIG,
    "safe_improving_configs": summary["safe_improving_configs"],
    "base_recent_brier": float(base_row["recent_brier"]),
    "selected_recent_brier": float(selected_row["recent_brier"]),
    "delta_recent_brier_vs_base": float(selected_row["delta_recent_brier_vs_base"]),
    "delta_brier_2023_vs_base": float(selected_row["delta_brier_2023_vs_base"]),
    "delta_brier_2024_vs_base": float(selected_row["delta_brier_2024_vs_base"]),
    "next_action": (
        "repeat_or_tune_selected"
        if summary["safe_improving_configs"] else "keep_base_and_move_to_trackman_mapping"
    ),
}
decision

### 12. 검토용 결과 ZIP 생성

이 ZIP은 DACON 제출물이 아니다. 결과 요약·폴드/구간/보정 지표·피처 목록과 중요도만 담는다. 대용량 OOF는 Drive의 실행 폴더에 그대로 두며, 우선 아래 ZIP과 실행 완료된 `05_feature_ablation_cv.ipynb`를 전달하면 다음 실험을 결정할 수 있다.

In [ ]:
review_names = [
    "summary.json",
    "configuration.json",
    "row_independence_audit.json",
    "config_summary.csv",
    "fold_metrics.csv",
    "group_metrics.csv",
    "calibration_tables.csv",
    "prediction_correlations.csv",
    "training_metadata.csv",
    "feature_manifest.csv",
    "feature_importance.csv",
]
missing_review = [name for name in review_names if not (RUN_DIR / name).exists()]
if missing_review:
    raise FileNotFoundError(f"검토용 결과가 누락되었습니다: {missing_review}")

REVIEW_ZIP = OUTPUT_ROOT / "feature_ablation_review.zip"
with zipfile.ZipFile(REVIEW_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in review_names:
        archive.write(RUN_DIR / name, arcname=name)

with zipfile.ZipFile(REVIEW_ZIP) as archive:
    print("review files:")
    for info in archive.infolist():
        print(f"  {info.filename}: {info.file_size / 1024:.1f} KB")
print("\n검토용 ZIP:", REVIEW_ZIP)
print("size (MB):", round(REVIEW_ZIP.stat().st_size / 1e6, 3))

### 13. 다음 단계

실행 완료 후 전달할 파일은 `feature_ablation_review.zip`과 출력이 저장된 `05_feature_ablation_cv.ipynb` 두 개다. 결과를 확인한 뒤 안정적으로 개선된 피처는 우리 모델에 반영하고, 이어서 공식 Trackman 데이터의 비식별 선수 연결 가능성을 검증하는 06 단계로 진행한다. 이번 실험 결과만으로는 제출 ZIP을 새로 만들지 않는다.